# Fine-Tuning Azure OpenAI model for Tool Calling

In [ ]:
%pip install -r requirements.txt  # install python dependencies

In [3]:
import os
import shutil

if not os.path.exists(".env"):
    shutil.copyfile(".env.template", ".env")

Update `.env` file with your Azure OpenAI resource details like endpoint and API key.

In [4]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [5]:
from openai import AzureOpenAI

# if os.environ fails with a KeyError, check whether the corresponding environment variable has been set in .env file
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]

client = AzureOpenAI(
    base_url=AZURE_OPENAI_ENDPOINT + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
    api_version=""
)

In [ ]:
MODEL = "gpt-4.1-mini"

[model for model in client.models.list() if model.id == MODEL]  # check whether client has been configured correctly

[Model(id='gpt-4.1-mini', created=None, object='model', owned_by=None, status='succeeded', capabilities={'fine_tune': True, 'inference': True, 'completion': False, 'chat_completion': True, 'embeddings': False, 'global_fine_tune': True}, lifecycle_status='generally-available', deprecation={'fine_tune': 1775865600, 'inference': 1775865600}, created_at=1744329600)]

## Tools Dataset

In [60]:
import json

TRAINING_FILE = "./data/drone_tools_train.jsonl"
VALIDATION_FILE = "./data/drone_tools_valid.jsonl"

with open(TRAINING_FILE) as f:
    first_line = f.readline().strip()
    drone_tools = json.loads(first_line)["tools"]

print("List of Tools and their Parameters available to the Drone:")
for tool in drone_tools:
    function_name = tool["function"]["name"]
    function_parameters = tool["function"]["parameters"]
    print(f"- {function_name:25}: {function_parameters}")

List of Tools and their Parameters available to the Drone:
- takeoff_drone            : {'type': 'object', 'properties': {'altitude': {'type': 'integer'}}, 'required': ['altitude']}
- land_drone               : {'type': 'object', 'properties': {'location': {'type': 'string', 'enum': ['current', 'home_base', 'custom']}, 'coordinates': {'type': 'object'}}, 'required': ['location']}
- control_drone_movement   : {'type': 'object', 'properties': {'direction': {'type': 'string', 'enum': ['forward', 'backward', 'left', 'right', 'up', 'down']}, 'distance': {'type': 'integer'}}, 'required': ['direction', 'distance']}
- set_drone_speed          : {'type': 'object', 'properties': {'speed': {'type': 'integer', 'minimum': 0}}, 'required': ['speed']}
- control_camera           : {'type': 'object', 'properties': {'mode': {'type': 'string', 'enum': ['photo', 'video', 'panorama']}, 'duration': {'type': 'integer'}}, 'required': ['mode']}
- control_gimbal           : {'type': 'object', 'properties': {'ti

## Fine-Tuning for Tool Calling

In [59]:
with open(TRAINING_FILE, "rb") as f:
    training_file = client.files.create(file=f, purpose="fine-tune")

with open(VALIDATION_FILE, "rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")


In [ ]:
finetuning_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model=MODEL,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 2
            }
        }
    },
    suffix="drone-tools"
)

In [27]:
import time

print(f"Waiting for finetuning job to complete: {finetuning_job.id}")
try:
    while True:
        finetuning_job = client.fine_tuning.jobs.retrieve(finetuning_job.id)
        if finetuning_job.status in ("succeeded", "failed", "cancelled"):
            print(f"Finetuning completed with '{finetuning_job.status}' status")
            break

        print(".", end="")
        time.sleep(15)
except KeyboardInterrupt:
    pass

Waiting for finetuning job to complete: ftjob-724408fea5344a9ba4d2f7d17e71517a
Finetuning completed with 'succeeded' status


## Deploying Fine-Tuned model

## Evaluating Fine-Tuned model against Base model